# Dataset Exploration — 600K Fitness Exercise & Workout Program Dataset

Two files:
- **`programs_detailed_boostcamp_kaggle.csv`** — 605,033 rows, one row per exercise per day per week per program
- **`program_summary.csv`** — 2,598 rows, one row per program

In [ ]:
import sys
from pathlib import Path
import ast
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..')))
from config.settings import EXERCISES_CSV, PROGRAMS_CSV

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load data

In [ ]:
detail = pd.read_csv(EXERCISES_CSV)
summary = pd.read_csv(PROGRAMS_CSV)

print(f'programs_detailed: {detail.shape[0]:,} rows x {detail.shape[1]} cols')
print(f'program_summary:   {summary.shape[0]:,} rows x {summary.shape[1]} cols')
print('\nprograms_detailed columns:', detail.columns.tolist())
print('program_summary columns:  ', summary.columns.tolist())

In [ ]:
detail.head(3)

In [ ]:
summary.head(3)

### Parse list-encoded columns
`level` and `goal` are stored as stringified Python lists — parse them into actual lists.

In [ ]:
def safe_parse_list(val):
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else [result]
    except Exception:
        return [val] if pd.notna(val) else []

for df in (detail, summary):
    df['level_list'] = df['level'].apply(safe_parse_list)
    df['goal_list']  = df['goal'].apply(safe_parse_list)

# Negative reps encode time-based exercises in seconds (e.g. -180 = 180 s)
detail['is_time_based'] = detail['reps'] < 0
detail['reps_abs']      = detail['reps'].abs()

print('Time-based exercise rows:', detail['is_time_based'].sum())
print('Sample goals (summary):', summary['goal_list'].iloc[0])

## 2. Missing values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in zip(axes, [detail, summary], ['programs_detailed', 'program_summary']):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=13)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    else:
        null_pct.plot(kind='barh', ax=ax, color='#e07b54')
        ax.set_xlabel('% missing')
        for bar, val in zip(ax.patches, null_pct):
            ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                    f'{val:.2f}%', va='center', fontsize=9)
    ax.set_title(f'Missing values — {title}')

plt.tight_layout()
plt.show()

## 3. Program-level analysis (program_summary)

### 3a. Goal distribution

In [ ]:
goal_counts = Counter(g for goals in summary['goal_list'] for g in goals)
goal_df = pd.DataFrame(goal_counts.most_common(), columns=['goal', 'count'])

fig, ax = plt.subplots(figsize=(13, 6))
colors = sns.color_palette('muted', len(goal_df))
ax.barh(goal_df['goal'][::-1], goal_df['count'][::-1], color=colors)
for bar in ax.patches:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
ax.set_xlabel('Number of programs')
ax.set_title('Goal distribution (multi-label — programs can have multiple goals)')
plt.tight_layout()
plt.show()
print(goal_df.to_string(index=False))

### 3b. Difficulty level distribution

In [ ]:
level_counts = Counter(l for levels in summary['level_list'] for l in levels)
level_df = pd.DataFrame(level_counts.most_common(), columns=['level', 'count'])

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(level_df['level'], level_df['count'],
              color=sns.color_palette('Blues_d', len(level_df))[::-1])
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
ax.set_ylabel('Count')
ax.set_title('Difficulty level distribution (multi-label)')
plt.tight_layout()
plt.show()

### 3c. Equipment distribution

In [ ]:
equip_counts = summary['equipment'].value_counts(dropna=True)
colors = sns.color_palette('Set2', len(equip_counts))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(equip_counts.index[::-1], equip_counts.values[::-1], color=colors)
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 f'{int(bar.get_width()):,}', va='center', fontsize=9)
axes[0].set_title('Equipment type (program count)')
axes[0].set_xlabel('Programs')

axes[1].pie(equip_counts.values, labels=equip_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Equipment share')
plt.tight_layout()
plt.show()

### 3d. Program length (weeks)

In [ ]:
pl = summary['program_length'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(pl, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Program length (weeks)')
axes[0].set_ylabel('Programs')
axes[0].set_title('Program length distribution')

pl_vc = pl.value_counts().sort_index().head(25)
axes[1].bar(pl_vc.index.astype(int), pl_vc.values, color='steelblue')
axes[1].set_xlabel('Weeks')
axes[1].set_ylabel('Programs')
axes[1].set_title('Top program lengths (exact counts, first 25)')

plt.tight_layout()
plt.show()
print(pl.describe())

### 3e. Time per workout distribution

In [ ]:
tpw = summary['time_per_workout'].dropna()
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(tpw, bins=50, color='#5e9ea0', edgecolor='white')
ax.axvline(tpw.median(), color='tomato',  linestyle='--', label=f'Median: {tpw.median():.0f} min')
ax.axvline(tpw.mean(),   color='orange',  linestyle='--', label=f'Mean:   {tpw.mean():.0f} min')
ax.set_xlabel('Minutes per workout')
ax.set_ylabel('Programs')
ax.set_title('Time per workout distribution')
ax.legend()
plt.tight_layout()
plt.show()
print(tpw.describe())

### 3f. Total exercises per program

In [ ]:
te = summary['total_exercises'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(te, bins=60, color='#7b68ee', edgecolor='white')
axes[0].set_xlabel('Total exercises')
axes[0].set_ylabel('Programs')
axes[0].set_title('Total exercises per program')

axes[1].boxplot(te, vert=True, patch_artist=True, boxprops=dict(facecolor='#b0c4de'))
axes[1].set_ylabel('Total exercises')
axes[1].set_title('Boxplot — total exercises')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()
print(te.describe())

### 3g. Programs created over time

In [ ]:
summary['created_dt'] = pd.to_datetime(summary['created'], errors='coerce')
monthly = summary.set_index('created_dt').resample('ME').size()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(monthly.index, monthly.values, alpha=0.4, color='teal')
ax.plot(monthly.index, monthly.values, color='teal', linewidth=1.5)
ax.set_xlabel('Month')
ax.set_ylabel('Programs created')
ax.set_title('Programs created per month')
plt.tight_layout()
plt.show()

## 4. Exercise-level analysis (programs_detailed)

### 4a. Top 25 most common exercises

In [ ]:
top_ex = detail['exercise_name'].value_counts().head(25)

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(top_ex.index[::-1], top_ex.values[::-1],
        color=sns.color_palette('viridis', 25))
for bar in ax.patches:
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=8)
ax.set_xlabel('Occurrences across all programs')
ax.set_title('Top 25 most programmed exercises')
plt.tight_layout()
plt.show()

### 4b. Sets distribution

In [ ]:
sets = detail['sets'].dropna()
sets_clip = sets.clip(0, 20)
sets_vc = sets_clip.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(sets_vc.index.astype(int), sets_vc.values, color='#4682b4')
axes[0].set_xlabel('Sets per exercise')
axes[0].set_ylabel('Exercise rows')
axes[0].set_title('Sets distribution (capped at 20)')

axes[1].boxplot(sets, patch_artist=True, boxprops=dict(facecolor='#b0c4de'), vert=True)
axes[1].set_ylabel('Sets')
axes[1].set_title('Boxplot — sets')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()
print(sets.describe())

### 4c. Reps distribution

> **Note:** Negative `reps` values encode time-based exercises in seconds (e.g. -180 = 180 s). Shown separately.

In [ ]:
rep_based  = detail[~detail['is_time_based']]['reps'].dropna()
time_based = detail[detail['is_time_based']]['reps_abs'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(rep_based.clip(0, 50), bins=50, color='#6a9fb5', edgecolor='white')
axes[0].set_xlabel('Reps')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Rep-based  ({len(rep_based):,} rows, capped at 50)')

axes[1].hist(time_based.clip(0, 300), bins=40, color='#e07b54', edgecolor='white')
axes[1].set_xlabel('Duration (seconds)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Time-based  ({len(time_based):,} rows, capped at 300 s)')

plt.tight_layout()
plt.show()
print('Rep-based stats: ', rep_based.describe().round(1).to_dict())
print('Time-based stats:', time_based.describe().round(1).to_dict())

### 4d. Intensity distribution (0–10 scale)

In [ ]:
intensity = detail['intensity'].dropna().astype(int)
int_vc = intensity.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(int_vc.index, int_vc.values,
              color=sns.color_palette('RdYlGn_r', len(int_vc)))
for bar, val in zip(bars, int_vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{val:,}', ha='center', fontsize=9)
ax.set_xlabel('Intensity (0 = lowest, 10 = highest)')
ax.set_ylabel('Exercise rows')
ax.set_title('Intensity distribution')
ax.set_xticks(int_vc.index)
plt.tight_layout()
plt.show()

### 4e. Exercises per workout day

In [ ]:
ex_per_day = (
    detail.groupby(['title', 'week', 'day'])['exercise_name']
    .count()
    .reset_index(name='exercise_count')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(ex_per_day['exercise_count'].clip(0, 30), bins=30,
             color='#9370db', edgecolor='white')
axes[0].set_xlabel('Exercises per workout day')
axes[0].set_ylabel('Workout days')
axes[0].set_title('Exercises per day (capped at 30)')

axes[1].boxplot(ex_per_day['exercise_count'], patch_artist=True,
                boxprops=dict(facecolor='#d8b4fe'), vert=True)
axes[1].set_ylabel('Exercises per day')
axes[1].set_title('Boxplot — exercises per day')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()
print(ex_per_day['exercise_count'].describe().round(1))

### 4f. Equipment distribution in exercise rows

In [ ]:
detail_equip = detail['equipment'].value_counts()
colors = sns.color_palette('Set2', len(detail_equip))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(detail_equip.index[::-1], detail_equip.values[::-1], color=colors)
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{int(bar.get_width()):,}', va='center', fontsize=9)
axes[0].set_title('Equipment type — exercise rows')
axes[0].set_xlabel('Exercise rows')

axes[1].pie(detail_equip.values, labels=detail_equip.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Equipment share (exercise rows)')
plt.tight_layout()
plt.show()

## 5. Cross-analysis

### 5a. Goal x Equipment heatmap

In [ ]:
rows = [
    {'goal': g, 'equipment': row['equipment']}
    for _, row in summary.iterrows()
    for g in row['goal_list']
]
ge_df = pd.DataFrame(rows)
pivot = ge_df.pivot_table(index='goal', columns='equipment', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Programs: Goal x Equipment')
ax.set_xlabel('Equipment')
ax.set_ylabel('Goal')
plt.tight_layout()
plt.show()

### 5b. Difficulty level x Program length

In [ ]:
rows = [
    {'level': l, 'program_length': row['program_length']}
    for _, row in summary.iterrows()
    for l in row['level_list']
]
lp_df = pd.DataFrame(rows).dropna()
level_order = [l for l in ['Beginner', 'Novice', 'Intermediate', 'Advanced', 'Elite']
               if l in lp_df['level'].unique()]

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=lp_df, x='level', y='program_length', order=level_order,
            palette='Blues', ax=ax)
ax.set_xlabel('Difficulty level')
ax.set_ylabel('Program length (weeks)')
ax.set_title('Program length by difficulty level')
plt.tight_layout()
plt.show()

### 5c. Median sets & reps by equipment

In [ ]:
rep_only = detail[~detail['is_time_based']]
agg = (
    rep_only.groupby('equipment')
    .agg(median_sets=('sets', 'median'), median_reps=('reps_abs', 'median'), count=('exercise_name', 'count'))
    .reset_index()
    .sort_values('count', ascending=False)
)

colors = sns.color_palette('Set2', len(agg))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(agg['equipment'], agg['median_sets'], color=colors)
axes[0].set_ylabel('Median sets')
axes[0].set_title('Median sets by equipment')

axes[1].bar(agg['equipment'], agg['median_reps'], color=colors)
axes[1].set_ylabel('Median reps')
axes[1].set_title('Median reps by equipment')

plt.tight_layout()
plt.show()
print(agg.to_string(index=False))

### 5d. Top goals weighted by exercise volume

In [ ]:
detail_goals = detail['goal_list'].explode().value_counts().head(15)

fig, ax = plt.subplots(figsize=(13, 5))
ax.barh(detail_goals.index[::-1], detail_goals.values[::-1],
        color=sns.color_palette('muted', len(detail_goals)))
for bar in ax.patches:
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
ax.set_xlabel('Exercise rows')
ax.set_title('Top goals — weighted by exercise count per program')
plt.tight_layout()
plt.show()

## 6. Key takeaways for the pipeline

| Finding | Implication |
|---|---|
| `level` and `goal` are stringified Python lists | Parse with `ast.literal_eval` at chunk time |
| Negative `reps` = time in seconds (e.g. -180 = 180 s) | Store `abs(reps)` + `is_time_based` bool in Qdrant payload |
| Only 4 equipment values: Full Gym / Garage Gym / At Home / Dumbbell Only | Exact-match payload filter is reliable — no fuzzy matching needed |
| `intensity` is an integer 0–10 | Use as a payload field for difficulty-graded retrieval |
| `description` has ~0.14% nulls | Drop null-description rows at chunk time |
| `program_summary` has the narrative text Designer needs | Field-grouped chunk: `title + description + goal + level + equipment + program_length` per program row |
| `programs_detailed` is exercise-row-level | One JSONL record per row; include `title` as a foreign key to link back to its program |